# 03 - Data-Driven Feature Engineering

This notebook builds a pure feature-generation pipeline for OBD time-series.

Design goals:
- No labels
- No risk scoring
- No classification logic
- One row per sliding window for unsupervised and sequence-ready modeling

In [1]:
import numpy as np
import pandas as pd

from scipy.stats import skew, kurtosis
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

pd.set_option("display.max_columns", None)

DATA_PATH = "../data/processed/clean_obd_data.csv"
OUTPUT_PATH = "../data/features/feature_dataset.csv"

# We keep a 1 Hz modeling view while preserving 10 Hz source information upstream.
RESAMPLE_FREQ = "1s"
WINDOW_SIZE = 5
STRIDE = 2
MULTISCALE_WINDOW = 10
CORR_THRESHOLD = 0.9
EPS = 1e-6

In [2]:
df = pd.read_csv(DATA_PATH)

if "MAP" in df.columns and "map" not in df.columns:
    df = df.rename(columns={"MAP": "map"})

required_columns = [
    "trip_id",
    "time",
    "speed",
    "rpm",
    "throttle",
    "map",
    "coolant_temp"
]

missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("Input shape:", df.shape)
df.head()

Input shape: (2693824, 12)


,trip_id,time,gap_flag,speed,rpm,throttle,map,maf,pedal_d,coolant_temp,intake_temp,ambient_temp
0,1,0.000,0,0.0,0.0,89.0,96.0,0.91,14.1,31.0,22.0,21.0
1,1,0.091,0,0.0,0.0,89.0,96.0,0.91,14.1,31.0,22.0,21.0
2,1,0.181,0,0.0,0.0,89.0,96.0,0.91,14.1,31.0,22.0,21.0
3,1,0.272,0,0.0,0.0,89.0,96.0,0.91,14.1,31.0,22.0,21.0
4,1,0.370,0,0.0,0.0,89.0,96.0,0.91,14.1,31.0,22.0,21.0


In [3]:
def normalize_trip_time(trip: pd.DataFrame) -> pd.DataFrame:
    trip = trip.copy()

    t_num = pd.to_numeric(trip["time"], errors="coerce")
    numeric_ratio = t_num.notna().mean()

    # Prefer numeric seconds when available (typical for OBD elapsed time).
    if numeric_ratio >= 0.8:
        if t_num.notna().sum() == 0:
            t_num = pd.Series(np.arange(len(trip), dtype=float), index=trip.index)
        t_num = t_num.interpolate(limit_direction="both").ffill().bfill()
        base_ts = pd.Timestamp("1970-01-01")
        trip["time_dt"] = base_ts + pd.to_timedelta(t_num, unit="s")
        return trip

    parsed_dt = pd.to_datetime(trip["time"], errors="coerce")
    if parsed_dt.notna().sum() >= max(3, int(0.5 * len(parsed_dt))):
        trip["time_dt"] = parsed_dt
        return trip

    # Last-resort monotonic timeline if no valid parse is possible.
    fallback = pd.Series(np.arange(len(trip), dtype=float), index=trip.index)
    trip["time_dt"] = pd.Timestamp("1970-01-01") + pd.to_timedelta(fallback, unit="s")
    return trip


def clean_trip(trip: pd.DataFrame) -> pd.DataFrame:
    trip = trip.sort_values("time").copy()
    trip = trip.drop_duplicates(subset=["time"], keep="last")

    num_cols = ["speed", "rpm", "throttle", "map", "coolant_temp"]

    trip[num_cols] = trip[num_cols].ffill()
    trip[num_cols] = trip[num_cols].interpolate(limit=3, limit_direction="both")

    trip["speed"] = trip["speed"].clip(lower=0)
    trip["rpm"] = trip["rpm"].clip(lower=0, upper=8000)

    return trip


cleaned_trips = []
for trip_id, trip in tqdm(df.groupby("trip_id"), desc="Cleaning trips"):
    trip = normalize_trip_time(trip)
    trip = clean_trip(trip)
    trip["trip_id"] = trip_id
    cleaned_trips.append(trip)

clean_df = pd.concat(cleaned_trips, ignore_index=True)
print("After cleaning:", clean_df.shape)
clean_df.head()

Cleaning trips: 100%|██████████████████████████████████████████████████████████████████| 81/81 [00:02<00:00, 28.74it/s]


After cleaning: (2693824, 13)


,trip_id,time,gap_flag,speed,rpm,throttle,map,maf,pedal_d,coolant_temp,intake_temp,ambient_temp,time_dt
0,1,0.000,0,0.0,0.0,89.0,96.0,0.91,14.1,31.0,22.0,21.0,1970-01-01 00:00:00.000
1,1,0.091,0,0.0,0.0,89.0,96.0,0.91,14.1,31.0,22.0,21.0,1970-01-01 00:00:00.091
2,1,0.181,0,0.0,0.0,89.0,96.0,0.91,14.1,31.0,22.0,21.0,1970-01-01 00:00:00.181
3,1,0.272,0,0.0,0.0,89.0,96.0,0.91,14.1,31.0,22.0,21.0,1970-01-01 00:00:00.272
4,1,0.370,0,0.0,0.0,89.0,96.0,0.91,14.1,31.0,22.0,21.0,1970-01-01 00:00:00.370


In [4]:
signals = ["speed", "rpm", "throttle", "map", "coolant_temp"]


def resample_trip_to_1hz(trip: pd.DataFrame) -> pd.DataFrame:
    trip = trip.sort_values("time_dt").set_index("time_dt")

    resampled = trip[signals].resample(RESAMPLE_FREQ).mean()
    resampled["valid_point"] = resampled[signals].notna().all(axis=1).astype(float)

    resampled[signals] = resampled[signals].interpolate(limit=2, limit_direction="both")
    resampled[signals] = resampled[signals].ffill().bfill()

    resampled = resampled.reset_index().rename(columns={"time_dt": "time"})
    resampled["trip_id"] = trip["trip_id"].iloc[0]
    return resampled


resampled_trips = []
for trip_id, trip in tqdm(clean_df.groupby("trip_id"), desc="Resampling trips"):
    resampled_trips.append(resample_trip_to_1hz(trip))

resampled_df = pd.concat(resampled_trips, ignore_index=True)
print("Resampled shape:", resampled_df.shape)
resampled_df.head()

Resampling trips: 100%|████████████████████████████████████████████████████████████████| 81/81 [00:01<00:00, 49.75it/s]

Resampled shape: (251894, 8)


,time,speed,rpm,throttle,map,coolant_temp,valid_point,trip_id
0,1970-01-01 00:00:00,0.0,0.0,89.0,96.0,31.0,1.0,1
1,1970-01-01 00:00:01,0.0,0.0,89.0,96.0,31.0,1.0,1
2,1970-01-01 00:00:02,0.0,0.0,89.0,96.0,31.0,1.0,1
3,1970-01-01 00:00:03,0.0,0.0,89.0,96.0,31.0,1.0,1
4,1970-01-01 00:00:04,0.0,0.0,89.0,96.0,31.0,1.0,1


In [5]:
def safe_skew(x: np.ndarray) -> float:
    return 0.0 if np.std(x) < EPS else float(skew(x))


def safe_kurtosis(x: np.ndarray) -> float:
    return 0.0 if np.std(x) < EPS else float(kurtosis(x))


def safe_corr(a: pd.Series, b: pd.Series) -> float:
    if a.nunique() < 2 or b.nunique() < 2:
        return 0.0
    val = a.corr(b)
    return 0.0 if pd.isna(val) else float(val)


def safe_cov(a: pd.Series, b: pd.Series) -> float:
    if len(a) < 2 or len(b) < 2:
        return 0.0
    val = np.cov(a.to_numpy(dtype=float), b.to_numpy(dtype=float), ddof=0)[0, 1]
    return 0.0 if np.isnan(val) else float(val)


def zero_crossing_rate(x: np.ndarray) -> float:
    if len(x) < 2:
        return 0.0
    signs = np.sign(x)
    return float(np.sum(np.diff(signs) != 0) / max(1, len(x) - 1))


def oscillation_rate(x: np.ndarray) -> float:
    if len(x) < 3:
        return 0.0
    d = np.diff(x)
    d_sign = np.sign(d)
    return float(np.sum(np.diff(d_sign) != 0) / max(1, len(x) - 1))


def histogram_entropy(series: pd.Series) -> float:
    x = series.to_numpy(dtype=float)
    if len(x) < 4 or np.std(x) < EPS:
        return 0.0

    bins = max(3, min(15, len(x) // 2))
    counts, _ = np.histogram(x, bins=bins)
    counts = counts.astype(float)

    # Add a tiny floor to avoid zero-probability instability.
    counts = counts + EPS
    probs = counts / counts.sum()
    return float(-(probs * np.log(probs)).sum())


def fft_energy_entropy(series: pd.Series) -> tuple[float, float]:
    x = series.to_numpy(dtype=float)
    if len(x) < 4 or np.std(x) < EPS:
        return 0.0, 0.0

    # Detrend by removing mean before FFT.
    x = x - np.mean(x)
    spec = np.fft.rfft(x)
    power = np.abs(spec) ** 2

    energy = float(power.sum() / max(1, len(power)))
    p = power / (power.sum() + EPS)
    p = p + EPS
    p = p / p.sum()
    entropy = float(-(p * np.log(p)).sum())
    return energy, entropy


def zscore_series(series: pd.Series) -> pd.Series:
    return (series - series.mean()) / (series.std() + EPS)


def add_stats(features: dict, series: pd.Series, prefix: str) -> None:
    x = series.to_numpy(dtype=float)
    features[f"{prefix}_mean"] = float(np.mean(x))
    features[f"{prefix}_std"] = float(np.std(x))
    features[f"{prefix}_min"] = float(np.min(x))
    features[f"{prefix}_max"] = float(np.max(x))
    features[f"{prefix}_median"] = float(np.median(x))
    features[f"{prefix}_q25"] = float(np.percentile(x, 25))
    features[f"{prefix}_q75"] = float(np.percentile(x, 75))
    features[f"{prefix}_skew"] = safe_skew(x)
    features[f"{prefix}_kurt"] = safe_kurtosis(x)

In [6]:
def extract_window_features(
    window: pd.DataFrame,
    long_window: pd.DataFrame,
    window_id: int,
    trip_thresholds: dict
) -> dict:
    w = window.copy()

    # Temporal dynamics
    w["acceleration"] = w["speed"].diff().fillna(0.0)
    w["jerk"] = w["acceleration"].diff().fillna(0.0)
    w["rpm_diff"] = w["rpm"].diff().fillna(0.0)
    w["throttle_diff"] = w["throttle"].diff().fillna(0.0)
    w["speed_roll_var"] = w["speed"].rolling(3, min_periods=2).var().fillna(0.0)

    features = {
        "window_id": window_id,
        "trip_id": int(w["trip_id"].iloc[0]),
        "start_time": w["time"].iloc[0],
        "end_time": w["time"].iloc[-1],
        "window_duration": int(len(w)),
        "valid_data_ratio": float(w["valid_point"].mean())
    }

    # Core statistical features
    for sig in signals:
        add_stats(features, w[sig], sig)

    # Dynamic features
    accel_abs = w["acceleration"].abs()
    features["accel_mean"] = float(w["acceleration"].mean())
    features["accel_std"] = float(w["acceleration"].std())
    features["accel_max"] = float(w["acceleration"].max())
    features["accel_min"] = float(w["acceleration"].min())
    features["jerk_mean"] = float(w["jerk"].mean())
    features["jerk_std"] = float(w["jerk"].std())
    features["rpm_diff_mean"] = float(w["rpm_diff"].mean())
    features["throttle_diff_mean"] = float(w["throttle_diff"].mean())
    features["speed_roll_var_mean"] = float(w["speed_roll_var"].mean())
    features["accel_zero_crossing_rate"] = zero_crossing_rate(w["acceleration"].to_numpy())
    features["throttle_oscillation_rate"] = oscillation_rate(w["throttle"].to_numpy())

    # Direction-aware acceleration + intensity
    features["positive_acceleration_ratio"] = float((w["acceleration"] > 0).mean())
    features["negative_acceleration_ratio"] = float((w["acceleration"] < 0).mean())
    features["driving_intensity"] = float(np.mean(np.square(w["acceleration"].to_numpy(dtype=float))))

    # Event features with per-trip global thresholds (consistent across windows)
    acc_hi = trip_thresholds["acc_high_global"]
    acc_lo = trip_thresholds["acc_low_global"]
    throttle_spike_hi = trip_thresholds["throttle_spike_global"]
    speed_idle_q = trip_thresholds["speed_idle_global"]
    rpm_idle_hi = trip_thresholds["rpm_idle_high_global"]

    count_hard_acc = int((w["acceleration"] >= acc_hi).sum())
    count_hard_brake = int((w["acceleration"] <= acc_lo).sum())
    count_throttle_spike = int((w["throttle_diff"].abs() >= throttle_spike_hi).sum())
    count_idle_high_rpm = int(((w["speed"] <= speed_idle_q) & (w["rpm"] >= rpm_idle_hi)).sum())

    features["count_hard_acceleration"] = count_hard_acc
    features["count_hard_braking"] = count_hard_brake
    features["count_throttle_spikes"] = count_throttle_spike
    features["count_idle_high_rpm"] = count_idle_high_rpm

    window_len = max(1, len(w))
    features["ratio_hard_acceleration"] = float(count_hard_acc / window_len)
    features["ratio_hard_braking"] = float(count_hard_brake / window_len)
    features["ratio_throttle_spikes"] = float(count_throttle_spike / window_len)
    features["ratio_idle_high_rpm"] = float(count_idle_high_rpm / window_len)

    # Window-normalized ratio features (z-score inside window)
    speed_z = zscore_series(w["speed"])
    rpm_z = zscore_series(w["rpm"])
    throttle_z = zscore_series(w["throttle"])
    accel_z = zscore_series(w["acceleration"])

    rpm_over_speed_norm = rpm_z / (speed_z + EPS)
    speed_over_throttle_norm = speed_z / (throttle_z + EPS)
    accel_over_throttle_norm = accel_z / (throttle_z + EPS)
    rpm_over_throttle_norm = rpm_z / (throttle_z + EPS)

    features["rpm_over_speed_norm_mean"] = float(rpm_over_speed_norm.mean())
    features["rpm_over_speed_norm_std"] = float(rpm_over_speed_norm.std())
    features["speed_over_throttle_norm_mean"] = float(speed_over_throttle_norm.mean())
    features["speed_over_throttle_norm_std"] = float(speed_over_throttle_norm.std())
    features["accel_over_throttle_norm_mean"] = float(accel_over_throttle_norm.mean())
    features["accel_over_throttle_norm_std"] = float(accel_over_throttle_norm.std())
    features["rpm_over_throttle_norm_mean"] = float(rpm_over_throttle_norm.mean())
    features["rpm_over_throttle_norm_std"] = float(rpm_over_throttle_norm.std())

    # Correlation-based response features
    features["corr_throttle_speed"] = safe_corr(w["throttle"], w["speed"])
    features["corr_throttle_rpm"] = safe_corr(w["throttle"], w["rpm"])

    # Response-delay features (t -> t+1)
    if len(w) > 1:
        features["lag1_corr_throttle_speed"] = safe_corr(w["throttle"].iloc[:-1], w["speed"].iloc[1:])
        features["lag1_corr_throttle_rpm"] = safe_corr(w["throttle"].iloc[:-1], w["rpm"].iloc[1:])
    else:
        features["lag1_corr_throttle_speed"] = 0.0
        features["lag1_corr_throttle_rpm"] = 0.0

    # Engine + context features
    map_q85 = float(w["map"].quantile(0.85))
    coolant_q15 = float(w["coolant_temp"].quantile(0.15))
    engine_load_proxy = w["rpm"] * w["map"]

    features["pct_time_high_map"] = float((w["map"] > map_q85).mean())
    features["pct_time_low_coolant"] = float((w["coolant_temp"] < coolant_q15).mean())
    features["engine_load_proxy_mean"] = float(engine_load_proxy.mean())
    features["engine_load_proxy_std"] = float(engine_load_proxy.std())

    # Frequency-domain + entropy features on longer, aligned context window
    speed_fft_energy, speed_fft_entropy = fft_energy_entropy(long_window["speed"])
    throttle_fft_energy, throttle_fft_entropy = fft_energy_entropy(long_window["throttle"])

    features["speed_fft_energy"] = speed_fft_energy
    features["speed_fft_entropy"] = speed_fft_entropy
    features["throttle_fft_energy"] = throttle_fft_energy
    features["throttle_fft_entropy"] = throttle_fft_entropy

    features["speed_entropy"] = histogram_entropy(w["speed"])
    features["throttle_entropy"] = histogram_entropy(w["throttle"])
    features["rpm_entropy"] = histogram_entropy(w["rpm"])

    # Driver micro-behavior
    stable_acc_th = float(accel_abs.quantile(0.30))
    features["throttle_smoothness"] = float(1.0 / (1.0 + w["throttle"].std()))
    features["acceleration_aggressiveness"] = float(w["acceleration"].std())
    features["steady_speed_ratio"] = float((accel_abs <= stable_acc_th).mean())
    features["speed_variability_index"] = float(w["speed"].std() / (abs(w["speed"].mean()) + EPS))

    # Multi-signal interaction
    features["cov_speed_rpm"] = safe_cov(w["speed"], w["rpm"])
    features["cov_throttle_acceleration"] = safe_cov(w["throttle"], w["acceleration"])

    # Consistency feature
    features["consistency_index"] = float(np.mean([
        w["speed"].std(),
        w["throttle"].std(),
        w["acceleration"].std()
    ]))

    # Optional multi-scale (10-second context ending at current window end)
    ms = long_window.copy()
    ms["acceleration"] = ms["speed"].diff().fillna(0.0)
    features["ms10_speed_mean"] = float(ms["speed"].mean())
    features["ms10_speed_std"] = float(ms["speed"].std())
    features["ms10_rpm_mean"] = float(ms["rpm"].mean())
    features["ms10_rpm_std"] = float(ms["rpm"].std())
    features["ms10_throttle_mean"] = float(ms["throttle"].mean())
    features["ms10_throttle_std"] = float(ms["throttle"].std())
    features["ms10_accel_std"] = float(ms["acceleration"].std())

    return features


rows = []
window_id = 0
for trip_id, trip in tqdm(resampled_df.groupby("trip_id"), desc="Window extraction"):
    trip = trip.sort_values("time").reset_index(drop=True)
    if len(trip) < WINDOW_SIZE:
        continue

    # Per-trip global thresholds for event consistency across windows.
    trip_dyn = trip.copy()
    trip_dyn["acceleration"] = trip_dyn["speed"].diff().fillna(0.0)
    trip_dyn["throttle_diff"] = trip_dyn["throttle"].diff().fillna(0.0)

    trip_thresholds = {
        "acc_high_global": float(trip_dyn["acceleration"].quantile(0.90)),
        "acc_low_global": float(trip_dyn["acceleration"].quantile(0.10)),
        "throttle_spike_global": float(trip_dyn["throttle_diff"].abs().quantile(0.90)),
        "speed_idle_global": float(trip_dyn["speed"].quantile(0.20)),
        "rpm_idle_high_global": float(trip_dyn["rpm"].quantile(0.80))
    }

    for start in range(0, len(trip) - WINDOW_SIZE + 1, STRIDE):
        end = start + WINDOW_SIZE
        window = trip.iloc[start:end]
        long_start = max(0, end - MULTISCALE_WINDOW)
        long_window = trip.iloc[long_start:end]

        rows.append(extract_window_features(window, long_window, window_id, trip_thresholds))
        window_id += 1

feature_df = pd.DataFrame(rows)
feature_df = feature_df.sort_values(["trip_id", "start_time"]).reset_index(drop=True)
print("Window-level feature shape:", feature_df.shape)
feature_df.head()

Window extraction:   0%|                                                                        | 0/81 [00:00<?, ?it/s]C:\Users\mhasa\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\mhasa\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Users\mhasa\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\mhasa\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Users\mhasa\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C

Window-level feature shape: (125805, 110)


,window_id,trip_id,start_time,end_time,window_duration,valid_data_ratio,speed_mean,speed_std,speed_min,speed_max,speed_median,speed_q25,speed_q75,speed_skew,speed_kurt,rpm_mean,rpm_std,rpm_min,rpm_max,rpm_median,rpm_q25,rpm_q75,rpm_skew,rpm_kurt,throttle_mean,throttle_std,throttle_min,throttle_max,throttle_median,throttle_q25,throttle_q75,throttle_skew,throttle_kurt,map_mean,map_std,map_min,map_max,map_median,map_q25,map_q75,map_skew,map_kurt,coolant_temp_mean,coolant_temp_std,coolant_temp_min,coolant_temp_max,coolant_temp_median,coolant_temp_q25,coolant_temp_q75,coolant_temp_skew,coolant_temp_kurt,accel_mean,accel_std,accel_max,accel_min,jerk_mean,jerk_std,rpm_diff_mean,throttle_diff_mean,speed_roll_var_mean,accel_zero_crossing_rate,throttle_oscillation_rate,positive_acceleration_ratio,negative_acceleration_ratio,driving_intensity,count_hard_acceleration,count_hard_braking,count_throttle_spikes,count_idle_high_rpm,ratio_hard_acceleration,ratio_hard_braking,ratio_throttle_spikes,ratio_idle_high_rpm,rpm_over_speed_norm_mean,rpm_over_speed_norm_std,speed_over_throttle_norm_mean,speed_over_throttle_norm_std,accel_over_throttle_norm_mean,accel_over_throttle_norm_std,rpm_over_throttle_norm_mean,rpm_over_throttle_norm_std,corr_throttle_speed,corr_throttle_rpm,lag1_corr_throttle_speed,lag1_corr_throttle_rpm,pct_time_high_map,pct_time_low_coolant,engine_load_proxy_mean,engine_load_proxy_std,speed_fft_energy,speed_fft_entropy,throttle_fft_energy,throttle_fft_entropy,speed_entropy,throttle_entropy,rpm_entropy,throttle_smoothness,acceleration_aggressiveness,steady_speed_ratio,speed_variability_index,cov_speed_rpm,cov_throttle_acceleration,consistency_index,ms10_speed_mean,ms10_speed_std,ms10_rpm_mean,ms10_rpm_std,ms10_throttle_mean,ms10_throttle_std,ms10_accel_std
0,0,1,1970-01-01 00:00:00,1970-01-01 00:00:04,5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,89.0,0.0,89.0,89.0,89.0,89.0,89.0,0.0,0.0,96.000000,0.000000,96.0,96.000000,96.0,96.0,96.0,0.0,0.00,31.0,0.0,31.0,31.0,31.0,31.0,31.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,5,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,89.0,0.0,0.0
1,1,1,1970-01-01 00:00:02,1970-01-01 00:00:06,5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,89.0,0.0,89.0,89.0,89.0,89.0,89.0,0.0,0.0,96.000000,0.000000,96.0,96.000000,96.0,96.0,96.0,0.0,0.00,31.0,0.0,31.0,31.0,31.0,31.0,31.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,5,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,89.0,0.0,0.0
2,2,1,1970-01-01 00:00:04,1970-01-01 00:00:08,5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,89.0,0.0,89.0,89.0,89.0,89.0,89.0,0.0,0.0,96.000000,0.000000,96.0,96.000000,96.0,96.0,96.0,0.0,0.00,31.0,0.0,31.0,31.0,31.0,31.0,31.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,5,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,89.0,0.0,0.0
3,3,1,1970-01-01 00:00:06,1970-01-01 00:00:10,5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,89.0,0.0,89.0,89.0,89.0,89.0,89.0,0.0,0.0,96.181818,0.363636,96.0,96.909091,96.0,96.0,96.0,1.5,0.25,31.0,0.0,31.0,31.0,31.0,31.0,31.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,5,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,89.0,0.0,0.0
4,4,1,1970-01-01 00:00:08,1970-01-01 00:00:12,5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,89.0,0.0,89.0,89.0,89.0,89.0,89.0,0.0,0.0,96.181818,0.363636,96.0,96.909091,96.0,96.0,96.0,1.5,0.25,31.0,0.0,31.

In [7]:
metadata_cols = ["window_id", "trip_id", "start_time", "end_time"]
numeric_feature_cols = [c for c in feature_df.columns if c not in metadata_cols]

feature_df[numeric_feature_cols] = feature_df[numeric_feature_cols].replace([np.inf, -np.inf], np.nan)
feature_df[numeric_feature_cols] = feature_df[numeric_feature_cols].fillna(0.0)

# Feature distribution check before scaling
dist_summary = feature_df[numeric_feature_cols].agg(["mean", "std", "min", "max"]).T
print("Distribution summary (first 20 rows):")
print(dist_summary.head(20))

# Inspect highly skewed features before filtering
skew_series = feature_df[numeric_feature_cols].skew().abs().sort_values(ascending=False)
print("Top skewed features (abs skew):")
print(skew_series.head(10))

# Feature stability checks: remove constant and near-zero-variance features
constant_cols = [c for c in numeric_feature_cols if feature_df[c].nunique(dropna=False) <= 1]
variance_series = feature_df[numeric_feature_cols].var()
near_zero_var_cols = variance_series[variance_series <= 1e-8].index.tolist()

drop_stability_cols = sorted(set(constant_cols + near_zero_var_cols))
stable_feature_cols = [c for c in numeric_feature_cols if c not in drop_stability_cols]

feature_df["start_time"] = pd.to_datetime(feature_df["start_time"], errors="coerce")
feature_df = feature_df.sort_values("start_time").reset_index(drop=True)

# Split by trip_id (prevents leakage)
unique_trips = feature_df["trip_id"].unique()
split_idx = int(len(unique_trips) * 0.8)

train_trips = set(unique_trips[:split_idx])

train_part = feature_df[feature_df["trip_id"].isin(train_trips)]

scaler = StandardScaler()
scaler.fit(train_part[stable_feature_cols])
scaled_values = scaler.transform(feature_df[stable_feature_cols])
scaled_cols = [f"{c}_scaled" for c in stable_feature_cols]
scaled_df = pd.DataFrame(scaled_values, columns=scaled_cols, index=feature_df.index)


def feature_group(col_name: str) -> str:
    base = col_name[:-7] if col_name.endswith("_scaled") else col_name
    if base.startswith("ms10_"):
        return "multiscale"
    if "fft" in base:
        return "frequency"
    if "entropy" in base:
        return "entropy"
    if base.startswith("count_") or base.startswith("ratio_"):
        return "events"
    if base.startswith("corr_") or base.startswith("lag1_"):
        return "response"
    if base.startswith("cov_"):
        return "interaction"
    if base.startswith("accel_") or base.startswith("jerk_") or base.startswith("rpm_diff") or base.startswith("throttle_diff"):
        return "temporal"
    return base.split("_")[0]


# Correlation filtering with group-aware retention
corr_matrix = scaled_df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
drop_candidates = [col for col in upper.columns if (upper[col] > CORR_THRESHOLD).any()]

selected_set = set(scaled_cols)
group_counts = {}
for col in scaled_cols:
    grp = feature_group(col)
    group_counts[grp] = group_counts.get(grp, 0) + 1

for col in drop_candidates:
    if col not in selected_set:
        continue

    grp = feature_group(col)

    # Find columns highly correlated with this column
    correlated_cols = upper.index[upper[col] > CORR_THRESHOLD].tolist()

    for other in correlated_cols:
        if other not in selected_set or other == col:
            continue

        base_col = col[:-7]
        base_other = other[:-7]

        # Compare variance of original (unscaled) features
        if feature_df[base_col].var() < feature_df[base_other].var():
            if group_counts.get(grp, 0) > 1:
                selected_set.remove(col)
                group_counts[grp] -= 1
                break 

selected_scaled_cols = [c for c in scaled_cols if c in selected_set]

# Final dataset: metadata + selected scaled features only
final_df = pd.concat([feature_df[metadata_cols], scaled_df[selected_scaled_cols]], axis=1)

# Final validation
if final_df[selected_scaled_cols].isna().any().any():
    raise ValueError("NaN detected in final scaled features")

if np.isinf(final_df[selected_scaled_cols].to_numpy(dtype=float)).any():
    raise ValueError("Infinite values detected in final scaled features")

non_numeric_cols = [
    c for c in selected_scaled_cols
    if not pd.api.types.is_numeric_dtype(final_df[c])
]
if non_numeric_cols:
    raise ValueError(f"Non-numeric selected features detected: {non_numeric_cols}")

print("Raw numeric features:", len(numeric_feature_cols))
print("Stable numeric features:", len(stable_feature_cols))
print("Dropped (stability):", len(drop_stability_cols))
print("Selected scaled features:", len(selected_scaled_cols))
print("Final dataset shape:", final_df.shape)

Distribution summary (first 20 rows):
                         mean         std       min          max
window_duration      5.000000    0.000000  5.000000     5.000000
valid_data_ratio     0.990881    0.093062  0.000000     1.000000
speed_mean          62.737815   45.771290  0.000000   200.000000
speed_std            1.496880    1.865209  0.000000    51.929183
speed_min           60.717408   46.174378  0.000000   200.000000
speed_max           64.767716   45.411786  0.000000   200.000000
speed_median        62.733823   45.827455  0.000000   200.000000
speed_q25           61.708290   46.015702  0.000000   200.000000
speed_q75           63.761838   45.622782  0.000000   200.000000
speed_skew           0.007884    0.638608 -1.500000     1.500000
speed_kurt          -0.805154    0.661371 -1.833333     0.250000
rpm_mean          1499.442307  520.642738  0.000000  3658.680000
rpm_std             58.264247   90.395603  0.000000  1047.356188
rpm_min           1424.553560  538.296556  0.000000 

In [8]:
final_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved feature dataset to: {OUTPUT_PATH}")

summary = pd.DataFrame({
    "metric": [
        "n_windows",
        "n_trips",
        "n_raw_numeric_features",
        "n_stable_numeric_features",
        "n_selected_scaled_features",
        "n_total_columns"
    ],
    "value": [
        len(final_df),
        final_df["trip_id"].nunique(),
        len(numeric_feature_cols),
        len(stable_feature_cols),
        len(selected_scaled_cols),
        final_df.shape[1]
    ]
})
summary

Saved feature dataset to: ../data/features/feature_dataset.csv


,metric,value
0,n_windows,125805
1,n_trips,81
2,n_raw_numeric_features,106
3,n_stable_numeric_features,105
4,n_selected_scaled_features,78
5,n_total_columns,82
